In [1]:
#Imports
import pickle
import os
from pathlib import Path
from shared.utils import run_model, load_snapshot
from shared.plotting import plot_sankey
from energyscope.models import Model

# 2050 - Sans pointe

In [2]:
# 1. Définition des chemins absolus
_BASE_DIR = Path(r"C:\Users\julie\Desktop\EnergyScope-Quebec")
_MODEL_2050_DIR = _BASE_DIR / 'shared' / 'model'
_DATA_2050_DIR = _BASE_DIR / 'shared' / 'data'
_DATA_EUD_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'EUD'
_DATA_TECHS_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'Techs'
_DATA_SHARES_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'Shares'
_PROJECT_DIR = _BASE_DIR / 'projects' / 'peaks' / '02_Model'

# 2. Construction du modèle unique ordonné (2050 Carboneutre - Sans Pointes, Avec HP)
energyscope_2050 = Model([
    # --- FICHIERS DE BASE (Partagés) ---
    ('mod', str(_MODEL_2050_DIR / 'QC_es_main.mod')),
    ('mod', str(_MODEL_2050_DIR / 'QC_objective_function.mod')),
    ('dat', str(_DATA_2050_DIR / 'QC_set_snapshot.dat')),
    ('dat', str(_DATA_2050_DIR / 'QC_data.dat')),
    ('dat', str(_DATA_EUD_2050_DIR / 'QC_eud_2050.dat')),
    ('dat', str(_DATA_SHARES_2050_DIR / 'QC_shares_2050.dat')),
    ('dat', str(_DATA_TECHS_2050_DIR / 'QC_techs_2050.dat')),

    # --- ÉLÉMENTS DE MODÉLISATION SUPPLÉMENTAIRES (.mod) ---
    ('mod', str(_PROJECT_DIR / 'HP_Winter.mod')),
    ('dat', str(_PROJECT_DIR / 'HP_Winter.dat')),

    #('mod', str(_PROJECT_DIR / 'Hydro_limit_annual.mod')), # Limite annuelle d'hydro
    #('mod', str(_PROJECT_DIR / 'imports_split_2050.mod')),
    #('mod', str(_PROJECT_DIR / 'realisme_2050.mod')),

    # --- DONNÉES ET SCRIPTS DE CONTEXTE ---
    # 1. Données spécifiques au scénario carboneutre (ex: co2_limit := 0;)
    #('dat', str(_PROJECT_DIR / 'carboneutre.dat')),
    #('dat', str(_PROJECT_DIR / 'imports_split_2050.dat')),
])

In [3]:
results_wo_peaks_2050 = run_model(energyscope_2050, apply_postprocessing=True)
plot_sankey(results_wo_peaks_2050)

Gurobi 12.0.3: 

In [4]:
output_dir = '../03_Results'
output_path = os.path.join(output_dir, 'results_wo_peaks_2050.pkl')
os.makedirs(output_dir, exist_ok=True)

# Sauvegarde
with open(output_path, 'wb') as f:
    pickle.dump(results_wo_peaks_2050, f)

In [5]:
df = results_wo_peaks_2050.variables['F_Mult']

# Filtrer les lignes contenant "HP" (sensible à la casse)
df_filtre = df[df.index.str.contains('HP', na=False)]

print(df_filtre)

                       F_Mult  Run
index                             
CEMENT_PROD_HP       0.000000    0
DEC_HP_ELEC          0.001444    0
DEC_HP_ELEC_PEAK     0.000000    0
DEC_HP_ELEC_WINTER   0.001444    0
DEC_THHP_BIOGAS      0.000000    0
DEC_THHP_GAS         0.000000    0
DHN_HP_ELEC          0.000000    0
EHP_H2_GRID          6.601340    0
EHP_NG_GRID         48.692451    0
FOOD_PROD_HP         0.000000    0
HP_H2_GRID           9.381075    0
HP_NG_GRID          48.692451    0
HYDRO_GAS_CHP        0.000000    0
IND_HP_ELEC          0.000000    0
PAPER_MAKING_HP      0.000000    0
STEEL_MAKING_HP      0.000000    0


# 2050 - Sans pointe - Carboneutre

In [6]:
from pathlib import Path

# 1. Définition des chemins absolus
_BASE_DIR = Path(r"C:\Users\julie\Desktop\EnergyScope-Quebec")
_MODEL_2050_DIR = _BASE_DIR / 'shared' / 'model'
_DATA_2050_DIR = _BASE_DIR / 'shared' / 'data'
_DATA_EUD_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'EUD'
_DATA_TECHS_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'Techs'
_DATA_SHARES_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'Shares'
_PROJECT_DIR = _BASE_DIR / 'projects' / 'peaks' / '02_Model'

# 2. Construction du modèle unique ordonné (2050 Carboneutre - Sans Pointes, Avec HP)
energyscope_2050_carboneutre = Model([
    # --- 1. DÉCLARATIONS DES MODÈLES (Tous les .mod de base et extensions) ---
    ('mod', str(_MODEL_2050_DIR / 'QC_es_main.mod')),
    ('mod', str(_MODEL_2050_DIR / 'QC_objective_function.mod')),
    ('dat', str(_DATA_2050_DIR / 'QC_set_snapshot.dat')),
    ('dat', str(_DATA_2050_DIR / 'QC_data.dat')),
    ('dat', str(_DATA_EUD_2050_DIR / 'QC_eud_2050.dat')),
    ('dat', str(_DATA_SHARES_2050_DIR / 'QC_shares_2050.dat')),
    ('dat', str(_DATA_TECHS_2050_DIR / 'QC_techs_2050.dat')),

    # --- 2. AJOUTS HP, SPLIT IMPORTS,  (Via des commandes 'let') ---
    ('mod', str(_PROJECT_DIR / 'HP_Winter.mod')),
    ('dat', str(_PROJECT_DIR / 'HP_Winter.dat')),

    ('mod', str(_PROJECT_DIR / 'Hydro_limit_annual.mod')),
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.mod')),
    ('dat', str(_PROJECT_DIR / 'imports_split_2050.dat')),

    ('mod', str(_PROJECT_DIR / 'realisme_2050.mod')),

    # --- 3. Carboneutre ---
    ('dat', str(_PROJECT_DIR / 'carboneutre.dat')),
])

In [7]:
results_wo_peaks_2050_carboneutre = run_model(energyscope_2050_carboneutre,apply_postprocessing=True)
plot_sankey(results_wo_peaks_2050_carboneutre)

Gurobi 12.0.3: 

In [8]:
output_dir = '../03_Results'
output_path = os.path.join(output_dir, 'results_wo_peaks_2050_carboneutre.pkl')
os.makedirs(output_dir, exist_ok=True)

# Sauvegarde
with open(output_path, 'wb') as f:
    pickle.dump(results_wo_peaks_2050_carboneutre, f)

In [9]:
df = results_wo_peaks_2050_carboneutre.variables['F_Mult']

# Filtrer les lignes contenant "HP" (sensible à la casse)
df_filtre = df[df.index.str.contains('HP', na=False)]

print(df_filtre)

                       F_Mult  Run
index                             
CEMENT_PROD_HP       0.000000    0
DEC_HP_ELEC          0.002014    0
DEC_HP_ELEC_PEAK     0.000000    0
DEC_HP_ELEC_WINTER   0.002014    0
DEC_THHP_BIOGAS      0.000000    0
DEC_THHP_GAS         0.000000    0
DHN_HP_ELEC          0.000000    0
EHP_H2_GRID          0.216629    0
EHP_NG_GRID         36.531438    0
FOOD_PROD_HP         0.000000    0
HP_H2_GRID           2.668646    0
HP_NG_GRID          36.531438    0
HYDRO_GAS_CHP        0.000000    0
IND_HP_ELEC          0.000000    0
PAPER_MAKING_HP      0.000000    0
STEEL_MAKING_HP      0.000000    0


# 2050 - Sans pointe - Carboneutre - Plan HQ

In [10]:
from pathlib import Path

# 1. Définition des chemins absolus
_BASE_DIR = Path(r"C:\Users\julie\Desktop\EnergyScope-Quebec")
_MODEL_2050_DIR = _BASE_DIR / 'shared' / 'model'
_DATA_2050_DIR = _BASE_DIR / 'shared' / 'data'
_DATA_EUD_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'EUD'
_DATA_TECHS_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'Techs'
_DATA_SHARES_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'Shares'
_PROJECT_DIR = _BASE_DIR / 'projects' / 'peaks' / '02_Model'

# 2. Construction du modèle unique ordonné (2050 Carboneutre - Sans Pointes, Avec HP)
energyscope_2050_carboneutre_planHQ = Model([
    # --- 1. DÉCLARATIONS DES MODÈLES (Tous les .mod de base et extensions) ---
    ('mod', str(_MODEL_2050_DIR / 'QC_es_main.mod')),
    ('mod', str(_MODEL_2050_DIR / 'QC_objective_function.mod')),
    ('dat', str(_DATA_2050_DIR / 'QC_set_snapshot.dat')),
    ('dat', str(_DATA_2050_DIR / 'QC_data.dat')),
    ('dat', str(_DATA_EUD_2050_DIR / 'QC_eud_2050.dat')),
    ('dat', str(_DATA_SHARES_2050_DIR / 'QC_shares_2050.dat')),
    ('dat', str(_DATA_TECHS_2050_DIR / 'QC_techs_2050.dat')),

    # --- 2. AJOUTS HP, SPLIT IMPORTS,  (Via des commandes 'let') ---
    ('mod', str(_PROJECT_DIR / 'HP_Winter.mod')),
    ('dat', str(_PROJECT_DIR / 'HP_Winter.dat')),

    ('mod', str(_PROJECT_DIR / 'Hydro_limit_annual.mod')),
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.mod')),
    ('dat', str(_PROJECT_DIR / 'imports_split_2050.dat')),

    ('mod', str(_PROJECT_DIR / 'realisme_2050.mod')),

    ('mod', str(_PROJECT_DIR / 'plan_action_HQ_2035.mod')),

    # --- 3. Carboneutre ---
    ('dat', str(_PROJECT_DIR / 'carboneutre.dat')),
])

In [11]:
results_wo_peaks_2050_carboneutre_planHQ = run_model(energyscope_2050_carboneutre_planHQ,apply_postprocessing=True)
plot_sankey(results_wo_peaks_2050_carboneutre_planHQ)

Gurobi 12.0.3: 

In [12]:
output_dir = '../03_Results'
output_path = os.path.join(output_dir, 'results_wo_peaks_2050_carboneutre_planHQ.pkl')
os.makedirs(output_dir, exist_ok=True)

# Sauvegarde
with open(output_path, 'wb') as f:
    pickle.dump(results_wo_peaks_2050_carboneutre_planHQ, f)

# 2050 - Avec pointe - Carboneutre

In [13]:
from pathlib import Path

# 1. Définition des chemins absolus
_BASE_DIR = Path(r"C:\Users\julie\Desktop\EnergyScope-Quebec")
_MODEL_2050_DIR = _BASE_DIR / 'shared' / 'model'
_DATA_2050_DIR = _BASE_DIR / 'shared' / 'data'
_DATA_EUD_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'EUD'
_DATA_TECHS_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'Techs'
_DATA_SHARES_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'Shares'
_PROJECT_DIR = _BASE_DIR / 'projects' / 'peaks' / '02_Model'

# 2. Construction du modèle unique ordonné (2050 Carboneutre - Sans Pointes, Avec HP)
energyscope_2050_carboneutre_peaks = Model([
    # --- 1. DÉCLARATIONS DES MODÈLES (Tous les .mod de base et extensions) ---
    ('mod', str(_MODEL_2050_DIR / 'QC_es_main.mod')),
    ('mod', str(_MODEL_2050_DIR / 'QC_objective_function.mod')),
    ('mod', str(_PROJECT_DIR / 'peaks_extra.mod')),
    ('dat', str(_DATA_2050_DIR / 'QC_set_snapshot.dat')),
    ('dat', str(_DATA_2050_DIR / 'QC_data.dat')),
    ('dat', str(_DATA_EUD_2050_DIR / 'QC_eud_2050.dat')),
    ('dat', str(_DATA_SHARES_2050_DIR / 'QC_shares_2050.dat')),
    ('dat', str(_DATA_TECHS_2050_DIR / 'QC_techs_2050.dat')),

    # --- 2. AJOUTS HP, SPLIT IMPORTS,  (Via des commandes 'let') ---

    ('mod', str(_PROJECT_DIR / 'HP_Winter.mod')),
    ('dat', str(_PROJECT_DIR / 'HP_Winter.dat')),

    ('mod', str(_PROJECT_DIR / 'Hydro_limit_annual.mod')),
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.mod')),
    ('dat', str(_PROJECT_DIR / 'imports_split_2050.dat')),

    ('mod', str(_PROJECT_DIR / 'realisme_2050.mod')),
    ('dat', str(_PROJECT_DIR / 'realisme_2050.dat')),
    #('mod', str(_PROJECT_DIR / 'plan_action_HQ_2035.mod')),

    # --- 3. Carboneutre ---
    ('dat', str(_PROJECT_DIR / 'carboneutre.dat')),

    # --- 4. Pointes ---
    ('dat', str(_PROJECT_DIR / 'peaks_2_2050.dat')),
    ('dat', str(_PROJECT_DIR / 'peaks_2050_cpt.dat')),
])

In [14]:
results_peaks_2050_carboneutre = run_model(energyscope_2050_carboneutre_peaks,apply_postprocessing=True)
plot_sankey(results_peaks_2050_carboneutre,aggregate_technology=False )

Gurobi 12.0.3: 

In [15]:
df = results_peaks_2050_carboneutre.variables['F_Mult']

# Filtrer les lignes contenant "HP" (sensible à la casse)
df_filtre = df[df.index.str.contains('HP', na=False)]

print(df_filtre)

                       F_Mult  Run
index                             
CEMENT_PROD_HP       0.000000    0
DEC_HP_ELEC          0.001391    0
DEC_HP_ELEC_PEAK     0.001391    0
DEC_HP_ELEC_WINTER   0.001391    0
DEC_THHP_BIOGAS      0.000000    0
DEC_THHP_GAS         0.000000    0
DHN_HP_ELEC          0.000000    0
EHP_H2_GRID          2.648540    0
EHP_NG_GRID         33.843652    0
FOOD_PROD_HP         0.000000    0
HP_H2_GRID           2.648540    0
HP_NG_GRID          33.843652    0
HYDRO_GAS_CHP        0.000000    0
IND_HP_ELEC          0.000000    0
PAPER_MAKING_HP      0.000000    0
STEEL_MAKING_HP      0.000000    0


In [16]:
df = results_peaks_2050_carboneutre.parameters['f_max']
df_filtre = df[df.index.str.contains('HP', na=False)]

print(df_filtre)

                      f_max  Run
index                           
CEMENT_PROD_HP      10000.0    0
DEC_HP_ELEC         10000.0    0
DEC_HP_ELEC_PEAK    10000.0    0
DEC_HP_ELEC_WINTER  10000.0    0
DEC_THHP_BIOGAS         0.0    0
DEC_THHP_GAS            0.0    0
DHN_HP_ELEC         10000.0    0
EHP_H2_GRID         10000.0    0
EHP_NG_GRID         10000.0    0
FOOD_PROD_HP        10000.0    0
HP_H2_GRID          10000.0    0
HP_NG_GRID          10000.0    0
HYDRO_GAS_CHP       10000.0    0
IND_HP_ELEC             0.0    0
PAPER_MAKING_HP     10000.0    0
STEEL_MAKING_HP     10000.0    0


In [17]:
output_dir = '../03_Results'
output_path = os.path.join(output_dir, 'results_peaks_2050_carboneutre.pkl')
os.makedirs(output_dir, exist_ok=True)

# Sauvegarde
with open(output_path, 'wb') as f:
    pickle.dump(results_peaks_2050_carboneutre, f)

# 2050 - Avec pointe - Carboneutre - Plan HQ

In [18]:
from pathlib import Path

# 1. Définition des chemins absolus
_BASE_DIR = Path(r"C:\Users\julie\Desktop\EnergyScope-Quebec")
_MODEL_2050_DIR = _BASE_DIR / 'shared' / 'model'
_DATA_2050_DIR = _BASE_DIR / 'shared' / 'data'
_DATA_EUD_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'EUD'
_DATA_TECHS_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'Techs'
_DATA_SHARES_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'Shares'
_PROJECT_DIR = _BASE_DIR / 'projects' / 'peaks' / '02_Model'

# 2. Construction du modèle unique ordonné (2050 Carboneutre - Sans Pointes, Avec HP)
energyscope_2050_carboneutre_peaks_planHQ = Model([
    # --- 1. DÉCLARATIONS DES MODÈLES (Tous les .mod de base et extensions) ---
    ('mod', str(_MODEL_2050_DIR / 'QC_es_main.mod')),
    ('mod', str(_MODEL_2050_DIR / 'QC_objective_function.mod')),
    ('mod', str(_PROJECT_DIR / 'peaks_extra.mod')),
    ('dat', str(_DATA_2050_DIR / 'QC_set_snapshot.dat')),
    ('dat', str(_DATA_2050_DIR / 'QC_data.dat')),
    ('dat', str(_DATA_EUD_2050_DIR / 'QC_eud_2050.dat')),
    ('dat', str(_DATA_SHARES_2050_DIR / 'QC_shares_2050.dat')),
    ('dat', str(_DATA_TECHS_2050_DIR / 'QC_techs_2050.dat')),

    # --- 2. AJOUTS HP, SPLIT IMPORTS,  (Via des commandes 'let') ---
    ('mod', str(_PROJECT_DIR / 'HP_Winter.mod')),
    ('dat', str(_PROJECT_DIR / 'HP_Winter.dat')),

    ('mod', str(_PROJECT_DIR / 'Hydro_limit_annual.mod')),
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.mod')),
    ('dat', str(_PROJECT_DIR / 'imports_split_2050.dat')),

    ('mod', str(_PROJECT_DIR / 'realisme_2050.mod')),
    ('dat', str(_PROJECT_DIR / 'realisme_2050.dat')),
    ('mod', str(_PROJECT_DIR / 'plan_action_HQ_2035.mod')),

    # --- 3. Carboneutre ---
    ('dat', str(_PROJECT_DIR / 'carboneutre.dat')),

    # --- 4. Pointes ---
    ('dat', str(_PROJECT_DIR / 'peaks_2_2050.dat')),
    ('dat', str(_PROJECT_DIR / 'peaks_2050_cpt.dat')),
])

In [19]:
results_peaks_2050_carboneutre_planHQ = run_model(energyscope_2050_carboneutre_peaks_planHQ,apply_postprocessing=True)
plot_sankey(results_peaks_2050_carboneutre_planHQ)

Gurobi 12.0.3: 

In [20]:
output_dir = '../03_Results'
output_path = os.path.join(output_dir, 'results_peaks_2050_carboneutre_planHQ.pkl')
os.makedirs(output_dir, exist_ok=True)

# Sauvegarde
with open(output_path, 'wb') as f:
    pickle.dump(results_peaks_2050_carboneutre_planHQ, f)

In [22]:
results_peaks_2050_carboneutre_planHQ

Result(constraints={}, parameters={'avail':                                 avail  Run
index                                     
ACETIC_ACID                       inf    0
ACETONE                           inf    0
BENZENE                           inf    0
BIOMASS_AGRICULTURE_DEJECTION  4149.0    0
BIOMASS_AGRICULTURE_RESIDUAL   2293.0    0
...                               ...  ...
WASTE_BIO                         0.0    0
WASTE_FOS                         0.0    0
WET_BIOMASS                       0.0    0
WOOD                              0.0    0
XYLENE                            inf    0

[86 rows x 2 columns], 'bio_ratio':    bio_ratio  Run
0          0    0, 'c_inv':                              c_inv  Run
index                                  
AFC                     629.639000    0
ALKALINE_ELECTROLYSIS  1274.606192    0
AL_MAKING                 0.000000    0
AL_MAKING_HR              0.000000    0
AN_DIG                 1807.811000    0
...                            ...

In [24]:
results_peaks_2050_carboneutre_planHQ.variables['F_Mult'].loc[['DEC_HP_ELEC']]

,F_Mult,Run
index,,
DEC_HP_ELEC,0.001391,0


In [25]:
df = results_peaks_2050_carboneutre_planHQ.parameters['layers_in_out'].loc['DEC_HP_ELEC_PEAK']
df_filtre = df[df['layers_in_out'] != 0]
df_filtre

,layers_in_out,Run
index1,,
ELECTRICITY_LV,-0.869565,0
HEAT_LOW_T_DECEN,1.000000,0


In [26]:
df = results_peaks_2050_carboneutre_planHQ.variables['F_Mult']
df

,F_Mult,Run
index,,
AFC,0.000000,0
ALKALINE_ELECTROLYSIS,0.000000,0
AL_MAKING,0.000000,0
AL_MAKING_HR,0.000000,0
AN_DIG,0.000000,0
...,...,...
TRUCK_SH_PROPANE_SD,0.000000,0
UNMINEABLE_COAL_SEAM,0.000000,0
WIND_OFFSHORE,0.000000,0


In [27]:
df = results_peaks_2050_carboneutre_planHQ.variables['F_Mult']

# Filtrer les lignes contenant "HP" (sensible à la casse)
df_filtre = df[df.index.str.contains('HP', na=False)]

print(df_filtre)

                       F_Mult  Run
index                             
CEMENT_PROD_HP       0.000000    0
DEC_HP_ELEC          0.001391    0
DEC_HP_ELEC_PEAK     0.001391    0
DEC_HP_ELEC_WINTER   0.001391    0
DEC_THHP_BIOGAS      0.000000    0
DEC_THHP_GAS         0.000000    0
DHN_HP_ELEC          0.000000    0
EHP_H2_GRID          2.648540    0
EHP_NG_GRID         33.843652    0
FOOD_PROD_HP         0.000000    0
HP_H2_GRID           2.648540    0
HP_NG_GRID          33.843652    0
HYDRO_GAS_CHP        0.000000    0
IND_HP_ELEC          0.000000    0
PAPER_MAKING_HP      0.000000    0
STEEL_MAKING_HP      0.000000    0


# 2050 - Sensibilité - Capture carbone

In [30]:
results_wo_peaks_2050_carboneutre.variables['F_Mult'].loc['CARBON_CAPTURE_MEMBRANES']

F_Mult    5.201681
Run       0.000000
Name: CARBON_CAPTURE_MEMBRANES, dtype: float64

# 2050 - Sensibilité - cout GN

In [23]:
from pathlib import Path
import pandas as pd
# Remplacer par le nom exact de votre module d'import de modèle local
# from ton_package import Model

# 1. Définition des chemins (Ton modèle de base)
_BASE_DIR = Path(r"C:\Users\julie\Desktop\EnergyScope-Quebec")
_MODEL_2050_DIR = _BASE_DIR / 'shared' / 'model'
_DATA_2050_DIR = _BASE_DIR / 'shared' / 'data'
_DATA_EUD_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'EUD'
_DATA_TECHS_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'Techs'
_DATA_SHARES_2050_DIR = _BASE_DIR / 'shared' / 'data' / 'Shares'
_PROJECT_DIR = _BASE_DIR / 'projects' / 'peaks' / '02_Model'

elements_modele = [
    ('mod', str(_MODEL_2050_DIR / 'QC_es_main.mod')),
    ('mod', str(_MODEL_2050_DIR / 'QC_objective_function.mod')),
    ('mod', str(_PROJECT_DIR / 'peaks_extra.mod')),
    ('dat', str(_DATA_2050_DIR / 'QC_set_snapshot.dat')),
    ('dat', str(_DATA_2050_DIR / 'QC_data.dat')),
    ('dat', str(_DATA_EUD_2050_DIR / 'QC_eud_2050.dat')),
    ('dat', str(_DATA_SHARES_2050_DIR / 'QC_shares_2050.dat')),
    ('dat', str(_DATA_TECHS_2050_DIR / 'QC_techs_2050.dat')),
    ('mod', str(_PROJECT_DIR / 'HP_Winter.mod')),
    ('dat', str(_PROJECT_DIR / 'HP_Winter.dat')),
    ('mod', str(_PROJECT_DIR / 'Hydro_limit_annual.mod')),
    ('mod', str(_PROJECT_DIR / 'imports_split_2050.mod')),
    ('dat', str(_PROJECT_DIR / 'imports_split_2050.dat')),
    ('mod', str(_PROJECT_DIR / 'realisme_2050.mod')),
    ('dat', str(_PROJECT_DIR / 'realisme_2050.dat')),
    ('dat', str(_PROJECT_DIR / 'carboneutre.dat')),
    ('dat', str(_PROJECT_DIR / 'peaks_2_2050.dat')),
    ('dat', str(_PROJECT_DIR / 'peaks_2050_cpt.dat')),
]

# 2. Définition de la plage de variation (ex: Multiplicateur du prix de base)
# On applique la variation uniformément sur les 12 mois de l'année 2050
multiplicateurs_prix = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]  # De -50% à +100% du prix initial
resultats_sensibilite = []

# 3. Boucle d'initialisation et de résolution
for mult in multiplicateurs_prix:
    print(f"\n>>> Scénario : Prix du Gaz Naturel à {mult * 100}% de sa valeur de base")

    # Réinitialisation propre à chaque itération
    instance_modele = Model(elements_modele)

    # Récupération de l'interface AMPL interne de votre objet Model
    ampl_interface = instance_modele.ampl

    # Optionnel: rendre le solveur silencieux (ici pour Gurobi)
    ampl_interface.set_option('solver', 'gurobi')
    ampl_interface.set_option('gurobi_options', 'outlev=0')

    # Commande AMPL pour faire varier c_op sur les 12 périodes de YEAR_2050 simultanément
    commande_let = f"let {{p in PERIODS}} c_op['YEAR_2050', 'NG_EHP', p] := c_op['YEAR_2050', 'NG_EHP', p] * {mult};"
    ampl_interface.eval(commande_let)

    # Résolution
    try:
        instance_modele.solve()

        # Extraction des résultats clés
        cout_total = ampl_interface.get_objective('TotalCost').value()
        gwp_total = ampl_interface.get_variable('TotalGWP').value()  # Optionnel: émission CO2 totale

        # Exemple d'extraction de la quantité totale de Gaz Naturel consommée (Annual_Res)
        qte_ng = ampl_interface.get_variable('Annual_Res').get('YEAR_2050', 'NG_EHP').value()

        print(f"   => Résolu avec succès. Coût global: {cout_total:.2f} | Consommation NG: {qte_ng:.2f}")

        resultats_sensibilite.append({
            'Multiplicateur_Prix_NG': mult,
            'Statut': 'Optimal',
            'TotalCost': cout_total,
            'TotalGWP': gwp_total,
            'Consommation_NG_GWh': qte_ng
        })

    except Exception as e:
        print(f"   => Échec de la résolution pour le multiplicateur {mult}: {e}")
        resultats_sensibilite.append({
            'Multiplicateur_Prix_NG': mult,
            'Statut': 'Infeasible/Erreur',
            'TotalCost': None,
            'TotalGWP': None,
            'Consommation_NG_GWh': None
        })

# 4. Traitement et sauvegarde des données générées
df_final = pd.DataFrame(resultats_sensibilite)
df_final.to_csv(_PROJECT_DIR / 'sensibilite_prix_gaz.csv', index=False)
print("\nAnalyse terminée. Fichier sauvegardé sous 'sensibilite_prix_gaz.csv'.")
print(df_final)


>>> Scénario : Prix du Gaz Naturel à 50.0% de sa valeur de base


AttributeError: 'Model' object has no attribute 'ampl'